# Avaliação final independente — Computer Vision
MobileNetV3 Small, CBIS-DDSM: benigno=0; maligno=1. Sem treinamento, fine-tuning ou seleção de limiar.

No Colab: **Arquivo → Fazer upload de notebook**, abra este arquivo e execute as células em ordem.
Envie `tech-challenge-cv-evaluation-code.zip` quando solicitado. Não envie novamente o dataset: usaremos o pacote CV-7 já no Drive.
Os resultados reais serão produzidos apenas por esta execução. O conjunto de teste não deve orientar novos ajustes.


In [ ]:
import sys
import importlib.metadata
print('Python:', sys.version)
for name in ('torch', 'torchvision', 'pandas', 'Pillow', 'matplotlib'):
    print(name, importlib.metadata.version(name))
import torch, torchvision
from torchvision.ops import nms
assert nms(torch.tensor([[0., 0., 1., 1.]]), torch.tensor([1.]), .5).numel() == 1
print('CUDA build:', torch.version.cuda, '| disponível:', torch.cuda.is_available())
print('Dispositivo:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


O Colab normalmente fornece essas bibliotecas. Esta célula não instala nem atualiza dependências.
Se a importação ou a operação NMS falhar, interrompa a execução e corrija a compatibilidade de PyTorch/torchvision em um runtime separado antes de continuar.
Não é necessário baixar pesos ImageNet: todos os parâmetros treinados serão restaurados do checkpoint.


In [ ]:
CODE_SOURCE = "upload_zip"
from google.colab import files
from pathlib import Path
import hashlib, os, tempfile, zipfile
EXPECTED_HASHES = {'src/__init__.py': 'b8fd4c32eed8c8f97c46a56f7a1caf7a0f58ccf919b8a8f629112054b260a299', 'src/vision/__init__.py': '84894d027a4e1d638dd623cb19f6d300c446a606186e211083fe39477a3eab02', 'src/vision/inspect_dataset.py': 'cdec84b8cf2865eeef34986617b7b02c1d3fcec2469b2ca171b9a0f49e312edb', 'src/vision/dataset.py': '28550d6f56067491b7946191f469eed6bee55ed214bc9bd2013e2b705f08799a', 'src/vision/model.py': 'e2247e95e7375074b7e301c2ac66603d3da3065df8ff61dddd602df126112c9e', 'src/vision/training.py': 'ac5a4bdec956c62de0d02dbb1484588a5b6a52ac722e29f6c489c86d5f686dc5', 'src/vision/evaluation.py': 'be95033e83d327468bd72312f69fddcb54ee2aded528978c478585c2bf6d8b14', 'tests/vision/test_evaluation.py': 'caaa5ba1911df8a5d18339b75cacc5c8437723dbe7c015ef26998dfa29f9a184'}
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Envie somente o ZIP de código atualizado.')
archive_path = next(iter(uploaded))
code_root = Path(tempfile.mkdtemp(prefix='cv_evaluation_code_'))
with zipfile.ZipFile(archive_path) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)) or set(names) != set(EXPECTED_HASHES):
        raise ValueError('ZIP incompatível: lista de arquivos inesperada.')
    if archive.testzip() is not None:
        raise ValueError('ZIP corrompido.')
    for name, expected in EXPECTED_HASHES.items():
        payload = archive.read(name)
        if hashlib.sha256(payload).hexdigest() != expected:
            raise ValueError('Código desatualizado ou alterado: ' + name)
        target = code_root / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(payload)
os.chdir(code_root)
sys.path.insert(0, str(code_root))
print('Código verificado:', code_root)


Execute os testes sintéticos de avaliação antes de acessar os dados reais. Nenhum teste exige download de pesos, GPU ou treinamento.
Skips não contam como validação efetiva: no Colab com PyTorch, todos devem executar.


In [ ]:
import unittest
suite = unittest.defaultTestLoader.discover('tests/vision', pattern='test_evaluation.py')
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful() and not result.skipped, 'Corrija falhas/skips antes da avaliação real.'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/FIAP-Tech-Challenge-Computer-Vision')
DATA_PACKAGE = BASE / 'cbis_ddsm_cv7.zip'
CHECKPOINT = BASE / 'artifacts/vision/run_001/best_epoch_004.pt'
# Diretório local temporário novo por execução; apenas teste será extraído.
TEST_DATA = Path(tempfile.mkdtemp(prefix='cv_final_test_')) / 'data'
# Nome único evita sobrescrever resultados e mantém run_001 intacto.
from datetime import datetime, timezone
OUTPUT_DIR = BASE / 'artifacts/vision' / ('evaluation_run_001_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
assert DATA_PACKAGE.is_file(), DATA_PACKAGE
assert CHECKPOINT.is_file(), CHECKPOINT
print('Checkpoint:', CHECKPOINT)
print('Pacote:', DATA_PACKAGE)
print('Resultados:', OUTPUT_DIR)


In [ ]:
from src.vision.evaluation import prepare_test_package
prepare_test_package(DATA_PACKAGE, TEST_DATA)
DATASET_ROOT = TEST_DATA / 'cbis_ddsm'
MANIFEST_DIR = TEST_DATA / 'processed'
import pandas as pd
manifest = pd.read_csv(MANIFEST_DIR / 'manifest_test.csv')
assert len(manifest) == 378
assert manifest['label'].value_counts().to_dict() == {0: 231, 1: 147}
assert set(manifest['split']) == {'test'} and set(manifest['official_split']) == {'test'}
print('Teste:', len(manifest), 'imagens; benignas=231, malignas=147')
print('Extração limitada ao teste; hashes das imagens conferidos.')


A próxima célula executa somente inferência: `eval()`, `inference_mode()` e argmax dos dois logits.
O carregamento é estrito e exige metadados CV-8 de treinamento/ImageNet verificado. A arquitetura é criada sem download e recebe integralmente o state_dict treinado antes de qualquer inferência.
Não usa treino/validação. O diretório de resultados deve ser novo. O SHA-256 identifica o checkpoint usado; os metadados não substituem uma cadeia externa de autenticidade do arquivo.


In [ ]:
from src.vision.evaluation import evaluate
metrics = evaluate(CHECKPOINT, DATASET_ROOT, MANIFEST_DIR, OUTPUT_DIR,
                   device='auto', batch_size=8)
assert metrics['checkpoint_epoch'] == 4, 'Época inesperada: confira o checkpoint.'
import json
print(json.dumps(metrics, ensure_ascii=False, indent=2))
from IPython.display import Image, Markdown, display
display(Image(filename=str(OUTPUT_DIR / 'confusion_matrix.png')))
display(Markdown((OUTPUT_DIR / 'evaluation_report.md').read_text(encoding='utf-8')))
print('Arquivos salvos:', [p.name for p in OUTPUT_DIR.iterdir()])


## Encerramento
Resultados: `metrics.json`, `confusion_matrix.png` e `evaluation_report.md`, com métricas e análise agregada dos falsos negativos.
Matriz: linhas = diagnóstico real, colunas = predição, na ordem benigno/maligno (`[[TN, FP], [FN, TP]]`).
Não são salvas imagens individuais nem informações de pacientes. As métricas são por recorte e não constituem validação clínica.
Preserve os resultados desta avaliação final para a apresentação; não use o teste para selecionar modelos ou ajustar hiperparâmetros.
